In [ ]:
EPOCHS = 15000
LR = 1E-4
BATCH_SIZE = 1

In [10]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")

Using MPS (Apple Silicon)
Device: mps


In [11]:
from radiance_field_project.data.datamodule import UCO3DDataModule

dm = UCO3DDataModule(
    batch_size=1,
    num_workers=0,
    image_height=512,
    image_width=512
)

train_loader = dm.train_loader()

print(f"Dataset size: {len(train_loader.dataset)}")
print(f"Num batches:  {len(train_loader)}")

Dataset size: 9399
Num batches:  9399


In [12]:
from radiance_field_project.models.simple_autoencoder import SimpleAutoencoder

network = SimpleAutoencoder()


In [13]:
import torch
from torch.optim import AdamW
import torch.nn.functional as F

from radiance_field_project.data.collate import collate_splats

optimizer = AdamW(network.parameters(), lr=LR)

torch.manual_seed(42)
sample = next(iter(train_loader))

features, batchidx = collate_splats(sample)
features.to(device)
print(features.shape)

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    recon = network(features)
    loss = F.mse_loss(features, recon)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Current loss = {loss.item():.6f}")





/Users/leonardkuch/Documents/Universitaet/Master/IS/ADL4CV/radiance-field-project/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


torch.Size([250000, 14])
Epoch 0: Current loss = 95.750801
Epoch 100: Current loss = 90.773270
Epoch 200: Current loss = 86.125511
Epoch 300: Current loss = 81.268257
Epoch 400: Current loss = 76.219650
Epoch 500: Current loss = 71.010521
Epoch 600: Current loss = 65.668335
Epoch 700: Current loss = 60.317066
Epoch 800: Current loss = 55.033787
Epoch 900: Current loss = 49.964573
Epoch 1000: Current loss = 45.130894
Epoch 1100: Current loss = 40.556515
Epoch 1200: Current loss = 36.240341
Epoch 1300: Current loss = 32.320732
Epoch 1400: Current loss = 28.786760
Epoch 1500: Current loss = 25.632710
Epoch 1600: Current loss = 22.836676
Epoch 1700: Current loss = 20.372393
Epoch 1800: Current loss = 18.209263
Epoch 1900: Current loss = 16.321096
Epoch 2000: Current loss = 14.674095
Epoch 2100: Current loss = 13.242874
Epoch 2200: Current loss = 12.008144
Epoch 2300: Current loss = 10.943095
Epoch 2400: Current loss = 10.023155
Epoch 2500: Current loss = 9.226291
Epoch 2600: Current loss =

KeyboardInterrupt: 

In [ ]:
from uco3d import render_splats
from radiance_field_project.data.collate import features_to_gaussians


render_colors_gt, render_alphas_gt, render_info_gt = render_splats(
    cameras=sample.camera,
    splats=sample.sequence_gaussian_splats,
    render_size=[512, 512]
)

recon = network(features)
sequence_gaussians_splats_reconstructed = features_to_gaussians(recon)

render_colors_recon, render_alphas_recon, render_info_recon = render_splats(
    cameras=sample.camera,
    splats=sequence_gaussians_splats_reconstructed,
    render_size=[512, 512]
)
